Notebook for downloading the forecasts submitted for the second sprint edition:

In [1]:
import numpy as np
import pandas as pd
import mosqlient as mosq
from epiweeks import Week
import matplotlib.pyplot as plt
from datetime import datetime, timedelta
from dotenv import load_dotenv
import os
load_dotenv()  # to load your api_key

True

Valid date interval according to the sprint rules: 

In [2]:
ref_dates_23 = set(pd.date_range(start= Week(2022, 41).startdate().strftime('%Y-%m-%d'),
              end= Week(2023, 40).startdate().strftime('%Y-%m-%d'),
              freq='W-SUN'))

ref_dates_24 = set(pd.date_range(start= Week(2023, 41).startdate().strftime('%Y-%m-%d'),
              end= Week(2024, 40).startdate().strftime('%Y-%m-%d'),
              freq='W-SUN'))

ref_dates_25 = set(pd.date_range(start= Week(2024, 41).startdate().strftime('%Y-%m-%d'),
              end= Week(2025, 40).startdate().strftime('%Y-%m-%d'),
              freq='W-SUN'))

ref_dates_26 = set(pd.date_range(start= Week(2025, 41).startdate().strftime('%Y-%m-%d'),
              end= Week(2026, 40).startdate().strftime('%Y-%m-%d'),
              freq='W-SUN'))

Get the predictions for a specific model:

In [3]:
def get_predictions_info(model_id):

    preds = mosq.get_predictions(api_key=api_key, model_id=model_id)

    df_predictions = pd.DataFrame()

    for pred_ in preds:
    
        preds_df = pred_.to_dataframe()
    
        preds_df.date = pd.to_datetime(preds_df.date)
    
        min_date = min(preds_df.date)
        
        max_date = max(preds_df.date)
    
        df_dates = set(preds_df.date)
    
        if min_date.year == 2022:
            ref_dates = ref_dates_23
            valid_test = 1 
            
        elif min_date.year == 2023:
            ref_dates = ref_dates_24
            valid_test = 2
    
        elif min_date.year == 2024:
            ref_dates = ref_dates_25
            valid_test = 3
    
        elif min_date.year == 2025:
            ref_dates = ref_dates_26
            valid_test = 'Forecast'
    
        missing_dates = ref_dates - df_dates
        extra_dates = df_dates - ref_dates
        
        if len(missing_dates) == 0:
            valid = True 
        else: 
            valid = False
    
        tem_zero = (preds_df[['lower_95', 'lower_90', 'lower_80', 'lower_50', 'pred',
           'upper_50', 'upper_80', 'upper_90', 'upper_95']] == 0).any().any()
    
        tem_negativo = ~(preds_df[['lower_95', 'lower_90', 'lower_80', 'lower_50', 'pred',
           'upper_50', 'upper_80', 'upper_90', 'upper_95']] < 0).any().any()
    
        quantile_order_strict = (
        (preds_df['lower_95'] < preds_df['lower_90']) &
        (preds_df['lower_90'] < preds_df['lower_80']) &
        (preds_df['lower_80'] < preds_df['lower_50']) &
        (preds_df['lower_50'] < preds_df['pred']) &
        (preds_df['pred']     < preds_df['upper_50']) &
        (preds_df['upper_50'] < preds_df['upper_80']) &
        (preds_df['upper_80'] < preds_df['upper_90']) &
        (preds_df['upper_90'] < preds_df['upper_95'])
        )
    
        quantile_order = (
        (preds_df['lower_95'] <= preds_df['lower_90']) &
        (preds_df['lower_90'] <= preds_df['lower_80']) &
        (preds_df['lower_80'] <= preds_df['lower_50']) &
        (preds_df['lower_50'] <= preds_df['pred']) &
        (preds_df['pred']     <= preds_df['upper_50']) &
        (preds_df['upper_50'] <= preds_df['upper_80']) &
        (preds_df['upper_80'] <= preds_df['upper_90']) &
        (preds_df['upper_90'] <= preds_df['upper_95'])
        )
         
        df_predictions = pd.concat([df_predictions, 
                                  pd.DataFrame(
                                      [[ pred_.id, 
                                        pred_.description,
                                        preds_df.shape[0], 
                                        min_date, 
                                        max_date, 
                                        pred_.dict()['adm_1'],
                                        tem_zero,
                                       tem_negativo,
                                        quantile_order.all(),
                                       quantile_order_strict.all(), 
                                       valid_test,
                                       valid, len(missing_dates), len(extra_dates)]], 
                                      columns = ['id',
                                                 'description', 
                                                 'size', 
                                                'start_date',
                                                'end_date', 'state', 
                                                'zero_values','nonnegative', 'q_order', 'qorder_stric',
                                                'validation_test', 'valid', 'missing_dates', 'extra_dates']) ], ignore_index = True)


    return df_predictions 
    

Get the forecasts:

In [4]:
%%time
label = 'Forecast'
dict_preds = {}
for model in [108, 133, 134, 135, 136,143, 144, 145, 150, 152, 154,155, 156, 157,  158]: 
    
    df_predictions = get_predictions_info(model)

    df_predictions = df_predictions.loc[((df_predictions.validation_test == 'Forecast') & (df_predictions.valid==True)) | 
((df_predictions.validation_test == 'Forecast') & ( (df_predictions.valid==False) & (df_predictions.missing_dates == 1) ))]

    if model == 135: 

        df_predictions = df_predictions.loc[df_predictions.description.str.contains('Sprint Rules')]

    dict_preds[model] = df_predictions.id.values


100%|███████████████████████████████████████| 3/3 [00:02<00:00,  1.11requests/s]


CPU times: user 6.68 s, sys: 378 ms, total: 7.06 s
Wall time: 1min 58s


In [5]:
%%time 
list_df_preds = [] 

for model in dict_preds.keys(): 

    for id_ in dict_preds[model]:

        if id_ == 4289:
            pass
            
        else:
            
            pred = mosq.get_prediction_by_id(api_key = api_key, id = id_ )

            df = pred.to_dataframe()

            df['model_id'] = model
            df['state'] = pred.dict()['adm_1']

            list_df_preds.append(df)

CPU times: user 15.3 s, sys: 3.42 s, total: 18.7 s
Wall time: 23min 22s


In [6]:
df_forecast = pd.concat(list_df_preds, axis =0, ignore_index = True)


df_forecast.groupby('model_id').state.nunique()

model_id
108    27
133    26
134    26
135    26
136    26
143    27
144    27
145    27
150    27
152    27
154    27
155    27
156    27
157    27
158    27
Name: state, dtype: int64

### Correct the predictions that contain only 52 weeks (instead of the 53 requested)

In [15]:
ref_dates_26 = set(pd.date_range(start= Week(2025, 41).startdate().strftime('%Y-%m-%d'),
              end= Week(2026, 40).startdate().strftime('%Y-%m-%d'),
              freq='W-SUN'))

In [20]:
df_forecast.date = pd.to_datetime(df_forecast.date)

In [59]:
df_debug = pd.DataFrame()
for model in df_forecast.model_id.unique():

    df_f = df_forecast.loc[df_forecast.model_id == model]

    for st in df_f.state.unique(): 

        df_f_st = df_f.loc[df_f.state == st]

        df_dates = set(df_f_st.date)

        missing_dates = ref_dates_26 - df_dates
        extra_dates = df_dates - ref_dates_26

        df_debug = pd.concat([df_debug, pd.DataFrame(
            [[model, st, len(missing_dates), len(extra_dates), missing_dates, extra_dates, df_f_st.shape[0]]], 
        columns = ['model', 'state', 'len_miss', 'len_extra', 'missing_dt', 'extra_dt', 'len_shape'])], ignore_index = True)


In [88]:
df_debug.len_shape.unique()

array([52, 53])

In [60]:
df_debug.len_miss.unique()

array([1, 0])

In [61]:
df_debug.len_extra.unique()

array([0])

In [65]:
df_debug.loc[df_debug.len_shape == 53].len_miss.unique()

array([0])

In [66]:
missing_dates = []
for v in df_debug.loc[df_debug.len_miss ==1 ].missing_dt:
    missing_dates.append(list(v)[0])

set(missing_dates)

{Timestamp('2025-12-28 00:00:00'), Timestamp('2026-10-04 00:00:00')}

Selecting predictions without errors:

In [71]:
preds_correct = list(df_debug.loc[(df_debug.len_shape == 53) & (df_debug.len_miss == 0) & (df_debug.len_extra == 0)][['model', 'state']].itertuples(index=False, name=None)) 

df_for_right = df_forecast[df_forecast[['model_id', 'state']].apply(tuple, axis =1).isin(preds_correct)]
df_for_right.head()

,date,lower_95,lower_90,lower_80,lower_50,pred,upper_50,upper_80,upper_90,upper_95,model_id,state
1404,2025-10-05,0.0,0.0,0.072723,9.686996,20.890854,44.136662,80.277618,94.786774,102.041351,133,TO
1405,2025-10-12,0.0,0.0,0.000000,9.369759,21.625626,45.486374,86.778954,103.449104,111.784180,133,TO
1406,2025-10-19,0.0,0.0,1.087540,11.811752,25.176056,52.454521,100.464600,119.861202,129.559504,133,TO
1407,2025-10-26,0.0,0.0,0.882904,10.941284,24.695343,54.576897,109.514786,131.941010,143.154121,133,TO
1408,2025-11-02,0.0,0.0,1.711456,12.073494,26.411240,59.000622,120.637886,145.808746,158.394176,133,TO


Selecting predictions with errors to be corrected:

In [111]:
import warnings 
warnings.simplefilter(action='ignore', category=pd.errors.SettingWithCopyWarning)


In [86]:
dict_rep_dates = {

    Week(2026, 1).startdate().strftime('%Y-%m-%d'): Week(2025, 53).startdate().strftime('%Y-%m-%d'),

    
}
for w in np.arange(2, 41): 

    dict_rep_dates[ Week(2026, w).startdate().strftime('%Y-%m-%d') ] = Week(2026, w-1).startdate().strftime('%Y-%m-%d')


In [113]:
df_for_corrected = pd.DataFrame()

for row in df_debug.loc[(df_debug.len_miss == 1) | (df_debug.len_shape == 52), ['model', 'state', 'missing_dt']].itertuples(index=False):
    model = row.model
    state = row.state
    miss_dt = row.missing_dt
    
    df_f = df_forecast.loc[(df_forecast.state == state) & (df_forecast.model_id == model)]

    if list(miss_dt)[0] == pd.to_datetime('2025-12-28'):

        df_f['date'] = df_f['date'].replace(dict_rep_dates)

        df_f = pd.concat([
            df_f,
            df_f.query("date == @pd.Timestamp('2026-09-27')").assign(date=pd.to_datetime('2026-10-04'))
        ], ignore_index=True)

    elif list(miss_dt)[0] == pd.to_datetime('2026-10-04'): 
        
        df_f = pd.concat([
            df_f,
            df_f.query("date == @pd.Timestamp('2026-09-27')").assign(date=pd.to_datetime('2026-10-04'))
        ], ignore_index=True)

    df_for_corrected = pd.concat([df_for_corrected, df_f])


Concatenate the dataframes: 

In [129]:
df_for_end = pd.concat([df_for_right, df_for_corrected], ignore_index = True)
df_for_end.date = pd.to_datetime(df_for_end.date)
df_for_end.head()

,date,lower_95,lower_90,lower_80,lower_50,pred,upper_50,upper_80,upper_90,upper_95,model_id,state
0,2025-10-05,0.0,0.0,0.072723,9.686996,20.890854,44.136662,80.277618,94.786774,102.041351,133,TO
1,2025-10-12,0.0,0.0,0.000000,9.369759,21.625626,45.486374,86.778954,103.449104,111.784180,133,TO
2,2025-10-19,0.0,0.0,1.087540,11.811752,25.176056,52.454521,100.464600,119.861202,129.559504,133,TO
3,2025-10-26,0.0,0.0,0.882904,10.941284,24.695343,54.576897,109.514786,131.941010,143.154121,133,TO
4,2025-11-02,0.0,0.0,1.711456,12.073494,26.411240,59.000622,120.637886,145.808746,158.394176,133,TO


In [152]:
# save the results
df_for_end.to_csv('predictions/forecasts_2nd_sprint_update.csv.gz', index = False)